In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Labs"
)

xpt_file = DATA_DIR / "TCHOL_H.xpt"
csv_file = DATA_DIR / "TCHOL_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 TCHOL_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXTC",
    "LBDTCSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TCHOL_H column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 8291:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,291 rows x 3 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# LBXTC:
#
#     Total cholesterol in mg/dL.
#     Released as whole-number values.
#
#
# LBDTCSI:
#
#     Total cholesterol in mmol/L.
#     Genuine decimal measurement.
#
# CRITICAL:
#
# DO NOT:
#
#     round LBDTCSI
#     convert LBDTCSI to Int64
#     globally strip ".0"
#     use float_format in to_csv()
#

DECIMAL_COLUMNS = [
    "LBDTCSI"
]


INTEGER_COLUMNS = [
    "SEQN",
    "LBXTC"
]


print("\n--- FIELD TYPE PLAN ---")


print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached TCHOL_H contains NONE.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all three TCHOL_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Tiny XPORT artifacts:",
    tiny_total
)


print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in TCHOL_H."
    )


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected zero values found "
        "in attached TCHOL_H."
    )


print(
    "PASS: No tiny artifacts and no ordinary zeros."
)


# ============================================================
# 6. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE converting whole-number fields.
#
# np.round() below is used ONLY for validation.
#
# It does NOT modify any data.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern differs.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "PASS: LBDTCSI is the only "
    "genuine decimal variable."
)


# ============================================================
# 7. DECIMAL FIELD DETAILS
# ============================================================

s = df["LBDTCSI"]


lbdtcsi_nonmissing = int(
    s.notna()
    .sum()
)


lbdtcsi_missing = int(
    s.isna()
    .sum()
)


lbdtcsi_min = float(
    s.dropna()
    .min()
)


lbdtcsi_max = float(
    s.dropna()
    .max()
)


values = (
    s.dropna()
    .to_numpy(
        dtype=float
    )
)


fractional_mask = ~np.isclose(
    values,
    np.round(values),
    rtol=0,
    atol=1e-12
)


lbdtcsi_fractional = int(
    fractional_mask.sum()
)


lbdtcsi_whole_looking = int(
    (~fractional_mask).sum()
)


print("\n--- LBDTCSI DETAILS ---")


print(
    "Nonmissing:",
    f"{lbdtcsi_nonmissing:,}"
)


print(
    "Missing:",
    f"{lbdtcsi_missing:,}"
)


print(
    "Fractional observations:",
    f"{lbdtcsi_fractional:,}"
)


print(
    "Whole-looking decimal observations:",
    f"{lbdtcsi_whole_looking:,}"
)


print(
    "Minimum:",
    lbdtcsi_min
)


print(
    "Maximum:",
    lbdtcsi_max
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 9. VALIDATE LBXTC AS WHOLE-NUMBER FIELD
# ============================================================

#
# Total cholesterol in mg/dL is released
# as whole-number values.
#

lbxtc_values = (
    df["LBXTC"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    lbxtc_values,
    np.round(lbxtc_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "LBXTC unexpectedly contains "
        "genuine fractional values."
    )


df["LBXTC"] = (
    df["LBXTC"]
    .astype("Int64")
)


print(
    "PASS: LBXTC safely converted "
    "to nullable Int64."
)


# ============================================================
# 10. CDC CHECK - LBXTC
# ============================================================

#
# Total Cholesterol - mg/dL
#
# CDC:
#
#     Range      = 69-813
#     Nonmissing = 7,624
#     Missing    = 667
#

s = df["LBXTC"]


lbxtc_nonmissing = int(
    s.notna()
    .sum()
)


lbxtc_missing = int(
    s.isna()
    .sum()
)


lbxtc_min = int(
    s.dropna()
    .min()
)


lbxtc_max = int(
    s.dropna()
    .max()
)


print("\n--- LBXTC ---")


print(
    "Nonmissing:",
    f"{lbxtc_nonmissing:,}"
)


print(
    "Missing:",
    f"{lbxtc_missing:,}"
)


print(
    "Minimum:",
    lbxtc_min
)


print(
    "Maximum:",
    lbxtc_max
)


if lbxtc_nonmissing != 7624:

    raise ValueError(
        "Unexpected LBXTC nonmissing count."
    )


if lbxtc_missing != 667:

    raise ValueError(
        "Unexpected LBXTC missing count."
    )


if lbxtc_min != 69:

    raise ValueError(
        "Unexpected LBXTC minimum."
    )


if lbxtc_max != 813:

    raise ValueError(
        "Unexpected LBXTC maximum."
    )


print(
    "PASS: LBXTC matches CDC range/count."
)


# ============================================================
# 11. CDC CHECK - LBDTCSI
# ============================================================

#
# Total Cholesterol - mmol/L
#
# CDC:
#
#     Range      = 1.78-21.02
#     Nonmissing = 7,624
#     Missing    = 667
#

if lbdtcsi_nonmissing != 7624:

    raise ValueError(
        "Unexpected LBDTCSI nonmissing count."
    )


if lbdtcsi_missing != 667:

    raise ValueError(
        "Unexpected LBDTCSI missing count."
    )


if not np.isclose(
    lbdtcsi_min,
    1.78,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDTCSI minimum."
    )


if not np.isclose(
    lbdtcsi_max,
    21.02,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected LBDTCSI maximum."
    )


print(
    "PASS: LBDTCSI matches CDC range/count."
)


# ============================================================
# 12. LBXTC / LBDTCSI MISSING-PATTERN CHECK
# ============================================================

#
# mg/dL and mmol/L are representations of
# the same laboratory result.
#
# Their missing positions should therefore match.
#

mg_nonmissing = (
    df["LBXTC"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


si_nonmissing = (
    df["LBDTCSI"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


missing_pattern_mismatch = int(
    np.sum(
        mg_nonmissing
        !=
        si_nonmissing
    )
)


print(
    "\n--- CHOLESTEROL MISSING-PATTERN CHECK ---"
)


print(
    "Mismatches:",
    missing_pattern_mismatch
)


if missing_pattern_mismatch != 0:

    raise ValueError(
        "LBXTC and LBDTCSI missing patterns differ."
    )


print(
    "PASS: LBXTC and LBDTCSI "
    "missing patterns match exactly."
)


# ============================================================
# 13. CDC SI-CONVERSION VALIDATION
# ============================================================

#
# CDC documentation:
#
#     LBDTCSI = LBXTC * 0.02586
#
# The released SI value is rounded to 2 decimals.
#
#
# IMPORTANT:
#
# Decimal + ROUND_HALF_UP is used ONLY for validating
# the CDC-derived SI value.
#
# We DO NOT overwrite LBDTCSI.
#
# We DO NOT recompute the CSV value.
#
# The released LBDTCSI value remains untouched.
#

paired = (
    df[
        [
            "LBXTC",
            "LBDTCSI"
        ]
    ]
    .dropna()
)


def calculate_expected_si(mg_dl):

    return float(
        (
            Decimal(
                str(
                    int(mg_dl)
                )
            )
            *
            Decimal("0.02586")
        )
        .quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )
    )


calculated_si = np.array(
    [
        calculate_expected_si(value)
        for value in paired["LBXTC"]
    ],
    dtype=float
)


released_si = (
    paired["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


si_mismatch_count = int(
    np.sum(
        calculated_si
        !=
        released_si
    )
)


print(
    "\n--- SI CONVERSION CHECK ---"
)


print(
    "Records checked:",
    f"{len(paired):,}"
)


print(
    "Conversion mismatches:",
    si_mismatch_count
)


if si_mismatch_count != 0:

    raise ValueError(
        "LBDTCSI does not match CDC's "
        "LBXTC x 0.02586 conversion."
    )


print(
    "PASS: All 7,624 released SI values match "
    "LBXTC x 0.02586 using 2-decimal "
    "ROUND_HALF_UP validation."
)


# ============================================================
# 14. IMPORTANT - DO NOT ALTER LBDTCSI
# ============================================================

#
# The SI calculation above is ONLY a validation.
#
# We intentionally DO NOT perform:
#
# df["LBDTCSI"] = calculated_si
#
# because the goal is XPT -> CSV fidelity.
#
# The released XPT value remains authoritative.
#

print(
    "PASS: LBDTCSI was validated, not recalculated."
)


# ============================================================
# 15. CONFIRM NO ZERO VALUES CREATED
# ============================================================

#
# Original attached file contained no zeros.
#
# Conversion should therefore not create any.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK ---"
)


print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values were created:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values created."
)


# ============================================================
# 16. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# TCHOL_H requires NO numeric corrections.
#
# Therefore every value must remain exactly
# numerically equal to the original XPT source.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# NO:
#
#     manual formatting of LBDTCSI
#
# Allow pandas to naturally serialize the genuine
# decimal field.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)


print(
    csv_file
)


# ============================================================
# 18. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 19. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 20. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 21. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 22. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 23. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT perform this check on LBDTCSI.
#
# LBDTCSI is a genuine decimal laboratory field.
#
# Only:
#
#     SEQN
#     LBXTC
#
# should be free of unnecessary ".0".
#

print(
    "\n--- INTEGER COLUMN .0 CHECK ---"
)


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in SEQN or LBXTC."
)


# ============================================================
# 24. LEGITIMATE LBDTCSI .0 VALUES
# ============================================================

#
# INFORMATIONAL ONLY.
#
# LBDTCSI is a genuine decimal measurement.
#
# Values ending in ".0" are therefore NOT errors.
#

lbdtcsi_dot_zero_count = int(
    (
        raw_csv_df["LBDTCSI"]
        .ne("")
        &
        raw_csv_df["LBDTCSI"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\n--- LEGITIMATE LBDTCSI .0 TOKENS ---"
)


print(
    "LBDTCSI values ending in '.0':",
    f"{lbdtcsi_dot_zero_count:,}"
)


if lbdtcsi_dot_zero_count != 47:

    raise ValueError(
        "Unexpected number of LBDTCSI '.0' tokens."
    )


print(
    "PASS: 47 legitimate decimal '.0' values "
    "were preserved."
)


# ============================================================
# 25. EXACT LBDTCSI DECIMAL FIDELITY
# ============================================================

expected_si = (
    source_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


exported_si = (
    roundtrip_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


same_si = np.allclose(
    expected_si,
    exported_si,
    rtol=0,
    atol=0,
    equal_nan=True
)


print(
    "\n--- EXACT LBDTCSI DECIMAL FIDELITY ---"
)


if not same_si:

    raise ValueError(
        "LBDTCSI decimal fidelity failed."
    )


print(
    "PASS: Every LBDTCSI decimal value "
    "preserved exactly."
)


# ============================================================
# 26. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zeros found in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "TCHOL_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Laboratory - Total Cholesterol"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "LBXTC nonmissing:",
    f"{lbxtc_nonmissing:,}"
)


print(
    "LBXTC range:",
    f"{lbxtc_min}-{lbxtc_max} mg/dL"
)


print(
    "LBDTCSI nonmissing:",
    f"{lbdtcsi_nonmissing:,}"
)


print(
    "LBDTCSI range:",
    f"{lbdtcsi_min}-{lbdtcsi_max} mmol/L"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


print(
    "Legitimate LBDTCSI '.0' values:",
    f"{lbdtcsi_dot_zero_count:,}"
)


print(
    "SI conversion mismatches:",
    si_mismatch_count
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "LBDTCSI decimal differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 TCHOL_H XPT -> CSV
Rows: 8,291
Columns: 3
PASS: Structure = 8,291 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal fields: ['LBDTCSI']
Whole-number fields: ['SEQN', 'LBXTC']

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: No tiny artifacts and no ordinary zeros.

--- GENUINE DECIMAL DETECTION ---
SEQN: 0 fractional observations
LBXTC: 0 fractional observations
LBDTCSI: 7,577 fractional observations
PASS: LBDTCSI is the only genuine decimal variable.

--- LBDTCSI DETAILS ---
Nonmissing: 7,624
Missing: 667
Fractional observations: 7,577
Whole-looking decimal observations: 47
Minimum: 1.78
Maximum: 21.02

--- SEQN ---
Minimum: 73557
Maximum: 83731
Missing: 0
Duplicates: 0
PASS: SEQN 73557-83731 validated.
PASS: LBXTC safely converted to nullable Int64.

--- LBXTC ---
Nonmissing: 7,624
Missing: 667
Minimum: 69
Maximum: 813
PASS: LBXTC matches CDC range/count.
PASS: LBDTCSI matches CDC range/count.

--- CHOLESTEROL MISSING-